<a target="_blank" href="https://colab.research.google.com/drive/1DzXfz2xG7G9lluMOoZtsWFS8Cf2-1U2h">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


### Github README.md Scraping

Repositories on github usually include a readme.md file that contains the description of the repository and some other important details.

In this notebook we will be using the github api to scrape readme.md files of some github repositories. This data will be suitable for topic modeling tasks.

We will start by installing the necessary libraries we need.

In [2]:
!pip install ghapi
!pip install pandas
!pip install beautifulsoup4
!pip install markdown
!pip install langdetect
!pip install nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=13655fc94901f57fd78c1d08136a8b088de589a76469a1a575140a419210f113
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


You can decide to generate an access token using your github account if your use case requires a higher rate limit, check this https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens to learn more about how to do that.

In [5]:
from ghapi.all import GhApi
import base64
import os

#  Provide the topics you are interesting in scraping
queries = [
    "artificial-intelligence",
    "devops"
]


def get_readmes(page, query, sort, order):
    api = GhApi()
    # Use the code below after generating your token
    # api = GhApi(token=token)

    # Make request to get first 10 repositories, save response in result array
    try:
      result = api.search.repos(q=query, sort=sort or "stars", order=order or "desc", page=page, per_page=10)
    except Exception as e:
      if hasattr(e, "code") and e.code == 403:
        print("Rate limit exceeded, generate and provide a token to continue")


    for repo in result["items"]:
      try:
        # make another request to extract readme files from repositories saved in result
        try:
          readme = api.repos.get_readme(owner=repo.owner.login, repo=repo.name)
        except Exception as e:
          if hasattr(e, "code") and e.code == 403:
            print("Rate limit exceeded, generate and provide a token to continue")

        # Save and write content of readme file
        if readme and readme.content:
            content = readme.content
            path = f"readmes/{query}"
            if not os.path.exists(path):
                os.makedirs(path)
            with open(
                f"{path}/{query.replace(' ', '_')}_{repo.owner.login}_{repo.name}_README.md",
                "w",
                encoding="utf-8",
            ) as f:
                f.write(base64.b64decode(content).decode("utf-8"))
      except Exception as e:
        print(f"Error processing repo {repo.owner.login}/{repo.name}: {e}")


for query in queries:
        get_readmes(1, query, "stars", "desc")

/usr/local/lib/python3.12/dist-packages/ghapi/core.py:106: UserWarning: Neither GITHUB_TOKEN nor GITHUB_JWT_TOKEN found: running as unauthenticated
  else: warn('Neither GITHUB_TOKEN nor GITHUB_JWT_TOKEN found: running as unauthenticated')


After successfully fetching the readme files from github using the github api, we will need to preprocess our data. We will use langdetect to filter out non english files, we will use BeautifulSoup and markdown to extract the text we need from the md files.

In [6]:
from bs4 import BeautifulSoup
from markdown import markdown
from langdetect import detect, LangDetectException
import re

count = 0

def is_english(text, min_length=50):
    try:
        return len(text.strip()) >= min_length and detect(text) == "en"
    except LangDetectException:
        return False

def remove_special_characters(text):
    text = re.sub(r'[^A-Za-z0-9\s]+', '', text)
    return text

def remove_urls(text):
    return re.sub(r'http\S+', '', text)

def remove_code_blocks(text):
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    return text

def remove_inline_code(text):
    text = re.sub(r'`.*?`', '', text)
    return text

def remove_markdown_links(text):
    return re.sub(r'\[.*?\]\(.*?\)', '', text)

def remove_markdown_characters(text):
    text = re.sub(r'[#*_~|>]+', '', text)
    return text

def remove_whitespace(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text

def clean_text(text):
    global count
    text = remove_code_blocks(text)  # Remove code blocks first
    text = remove_inline_code(text)  # Remove inline code
    text = remove_markdown_links(text)  # Remove markdown links
    text = remove_urls(text)

    # Convert Markdown to HTML
    html = markdown(text)
    # Use BeautifulSoup to extract text from HTML
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=' ', strip=True) # strip tags to plain text

    # Remove leftover markdown symbols
    text = remove_markdown_characters(text)
    text = remove_whitespace(text)

    # Check if readme is written in english
    if(not is_english(text)):
        count = count + 1
        print("Non-English text detected, skipping...")
        return ""

    text = remove_special_characters(text)

    return text.strip()

# Use this to extract text from md files in readme/ and save them to text_readmes/

def extract_text_from_readme(readmes_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    for filename in os.listdir(readmes_dir):
      if filename.endswith(".md"):
          src = os.path.join(readmes_dir, filename)
          dst = os.path.join(output_dir, filename.replace(".md", ".txt"))
          with open(src, "r", encoding="utf-8", errors="ignore") as f:
              text = clean_text(f.read())
              if(text != ""):
                with open(dst, "w", encoding="utf-8") as f:
                    f.write(text)

for query in queries:
    extract_text_from_readme(f"readmes/{query}", f"text_readmes/{query}")

print(f"Finished processing {query} readmes. Non-English count: {count}")

Non-English text detected, skipping...
Finished processing devops readmes. Non-English count: 1
